# Modelo digitalidad del cliente

In [0]:
!pip install kmodes scikit-learn-extra

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import col, lit, udf
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder.appName("Modelo Digitalidad del Cliente").getOrCreate()

In [0]:
df = spark.read.table("workspace.default.transaccional_clientes")
display(df.limit(10))

In [0]:
id_cols = ['cliente_id', 'agencia_id', 'ruta_id']

cat_cols = ['pais', 'region_comercial', 'tipo_cliente', 'madurez_digital', 'frecuencia_visitas', 'canal_pedido']

disc_cols = ['estrellas', 'materiales_distintos']

num_cols = ['facturacion_usd', 'cajas_fisicas']

date_cols = ['fecha_pedido_dt']

## Análisis exploratorio de datos

In [0]:
cliente_canal = (
    df
    .groupBy("cliente_id", "canal_pedido")
    .agg(f.count("*").alias("count"))
)

total_por_cliente = df.groupBy("cliente_id").agg(f.count("*").alias("total"))

cliente_canal = (
    cliente_canal
    .join(total_por_cliente, on="cliente_id", how="left")
    .withColumn("percentage", col("count") / col("total"))
    .select("cliente_id", "canal_pedido", "count", "percentage")
)

# Convert to pandas for plotting
cliente_canal_pd = cliente_canal.toPandas()

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=cliente_canal_pd,
    x="percentage",
    hue="canal_pedido"
)
plt.title("Histograma del porcentaje por canal_pedido")
plt.xlabel("Porcentaje")
plt.ylabel("Frecuencia")
plt.legend(title="Canal Pedido")
plt.show()

In [0]:
df_ym = df.withColumn(
    "year_month",
    f.date_format(col("fecha_pedido"), "yyyyMM").cast("int")
)

cliente_min_max = (
    df_ym.groupBy("cliente_id")
    .agg(
        f.min("year_month").alias("min_year_month"),
        f.max("year_month").alias("max_year_month"),
        f.countDistinct("year_month").alias("meses_activo")
    )
)

cliente_min_max = (
    cliente_min_max
    .withColumn("min_date", f.to_date(f.concat_ws("-", col("min_year_month").cast("string").substr(1,4), col("min_year_month").cast("string").substr(5,2), f.lit("01"))))
    .withColumn("max_date", f.to_date(f.concat_ws("-", col("max_year_month").cast("string").substr(1,4), col("max_year_month").cast("string").substr(5,2), f.lit("01"))))
    .withColumn("meses_total", f.months_between(col("max_date"), col("min_date")) + 1)
    .withColumn("meses_total", col("meses_total").cast("int"))
    .select("cliente_id", "min_year_month", "max_year_month", "meses_total", "meses_activo")
)

display(cliente_min_max.limit(20))

In [0]:
import random
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

clientes_all = [row['cliente_id'] for row in df.select("cliente_id").distinct().collect()]
n_clients = min(20, len(clientes_all))
clientes_sample = random.sample(clientes_all, n_clients)

wind_cliente_seq = Window.partitionBy("cliente_id").orderBy("fecha_pedido")

df_seq = (
    df.filter(col("cliente_id").isin(clientes_sample))
    .withColumn("nro_pedido", F.row_number().over(wind_cliente_seq))
    .withColumn("es_digital", (col("canal_pedido") == "DIGITAL").cast("int"))
    .select("cliente_id", "nro_pedido", "es_digital")
    .orderBy("cliente_id", "nro_pedido")
)

df_seq_pd = df_seq.toPandas()

num_clientes = len(clientes_sample)
fig, axes = plt.subplots(num_clientes, 1, figsize=(12, 3*num_clientes), sharex=True)

if num_clientes == 1:
    axes = [axes]

for ax, cliente in zip(axes, clientes_sample):
    data = df_seq_pd[df_seq_pd['cliente_id'] == cliente]
    ax.plot(
        data['nro_pedido'],
        data['es_digital'],
        marker='o',
        drawstyle='steps-mid',
        color='tab:blue'
    )
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['NO DIGITAL', 'DIGITAL'])
    ax.set_title(f'Cliente {cliente}')
    ax.set_ylabel('Canal Digital')

axes[-1].set_xlabel('Número de Pedido')
plt.tight_layout()
plt.show()

## Creación del dataset

In [0]:
df_cliente = (
    df
    .groupBy("cliente_id")
    .agg(
        f.last("agencia_id").alias("agencia_id"),
        f.last("ruta_id").alias("ruta_id"),
        f.last("pais").alias("pais"),
        f.last("region_comercial").alias("region_comercial"),
        f.last("tipo_cliente").alias("tipo_cliente"),
        f.last("madurez_digital").alias("madurez_digital"),
        f.last("estrellas").alias("estrellas"),
        f.last("frecuencia_visitas").alias("frecuencia_visitas"),
        f.sum("facturacion_usd").alias("sum_facturacion_usd"),
        f.avg("materiales_distintos").alias("avg_materiales_distintos"),
        f.sum("cajas_fisicas").alias("sum_cajas_fisicas"),
        f.sum((col("canal_pedido") == "DIGITAL").cast("int")).alias("pedidos_digitales"),
        f.count("*").alias("total_pedidos"),
        (f.sum((col("canal_pedido") == "DIGITAL").cast("int")) / f.count("*")).alias("proporcion_digital")
    )
)

display(df_cliente.limit(10))

In [0]:
df_pd = df_cliente.toPandas()

cat_cols = ['pais', 'region_comercial', 'tipo_cliente', 'frecuencia_visitas']
ordinal_cols = ['estrellas', 'madurez_digital']
num_cols = ['sum_facturacion_usd', 'avg_materiales_distintos', 'sum_cajas_fisicas', 'pedidos_digitales', 'total_pedidos']

for col_name in cat_cols:
    df_pd[col_name] = df_pd[col_name].astype("category")

df_pd['madurez_digital'] = pd.Categorical(
    df_pd['madurez_digital'],
    categories=["BAJA", "MEDIA", "ALTA"],
    ordered=True
)

df_pd['estrellas'] = pd.Categorical(
    df_pd['estrellas'],
    categories=[1, 2, 3],
    ordered=True
)

print('Dataset original:', f"{df.count():,}")
print('Dataset a nivel cliente:', f"{df_cliente.count():,}")
print('Dataset a nivel cliente (pandas):', f"{len(df_pd):,}")

display(df_pd)

## Análisis de variables

In [0]:
sns.histplot(data=df_pd, x='proporcion_digital', bins=15)

In [0]:
fig, axes = plt.subplots(len(cat_cols + ordinal_cols), 1, figsize=(10, 6 * len(cat_cols + ordinal_cols)))

for ax, col_name in zip(axes, cat_cols + ordinal_cols):
    sns.boxplot(
        data=df_pd,
        x=col_name,
        y="proporcion_digital",
        hue=col_name,
        ax=ax
    )
    ax.set_title(f"{col_name.replace('_', ' ').title()} vs Proporción Digital")
    ax.set_xlabel(col_name.replace('_', ' ').title())
    ax.set_ylabel("Proporción de Pedidos Digitales")
    ax.legend(title=col_name.replace('_', ' ').title(), loc='best')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(len(num_cols), 1, figsize=(10, 6 * len(num_cols)))

for ax, col_name in zip(axes, num_cols):
    sns.kdeplot(
        data=df_pd,
        x=col_name,
        y="proporcion_digital",
        fill=True,
        cmap="Blues",
        ax=ax
    )
    ax.set_title(f"{col_name.replace('_', ' ').title()} vs Proporción Digital")
    ax.set_xlabel(col_name.replace('_', ' ').title())
    ax.set_ylabel("Proporción de Pedidos Digitales")

plt.tight_layout()
plt.show()

## Solución

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

sns.boxplot(
    data=df_pd,
    x="madurez_digital",
    y="proporcion_digital",
    hue="total_pedidos",
    ax=ax
)
ax.set_title("Madurez Digital vs Proporción Digital")
ax.set_xlabel("Madurez Digital")
ax.set_ylabel("Proporción de Pedidos Digitales")

# Calcular los percentiles relevantes por grupo
q3_baja = df_pd[df_pd["madurez_digital"] == "BAJA"]["proporcion_digital"].quantile(0.75)
q1_media = df_pd[df_pd["madurez_digital"] == "MEDIA"]["proporcion_digital"].quantile(0.25)
q3_media = df_pd[df_pd["madurez_digital"] == "MEDIA"]["proporcion_digital"].quantile(0.75)
q1_alta = df_pd[df_pd["madurez_digital"] == "ALTA"]["proporcion_digital"].quantile(0.25)

frontera_baja_media = (q3_baja + q1_media) / 2
frontera_media_alta = (q3_media + q1_alta) / 2

ax.axhline(frontera_baja_media, color="gray", linestyle="--")
ax.text(
    ax.get_xlim()[1], frontera_baja_media, f"{frontera_baja_media:.2f}",
    color="gray", va="bottom", ha="right", fontsize=10, fontweight='bold'
)

ax.axhline(frontera_media_alta, color="gray", linestyle="--")
ax.text(
    ax.get_xlim()[1], frontera_media_alta, f"{frontera_media_alta:.2f}",
    color="gray", va="bottom", ha="right", fontsize=10, fontweight='bold'
)

plt.tight_layout()
plt.show()